In [3]:
pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   - -------------------------------------- 1.3/48.9 MB 19.4 MB/s eta 0:00:03
   ----- ---------------------------------- 6.8/48.9 MB 32.2 MB/s eta 0:00:02
   --------- ------------------------------ 11.5/48.9 MB 24.0 MB/s eta 0:00:02
   -------------------- ------------------- 25.2/48.9 MB 37.1 MB/s eta 0:00:01
   -------------------------------- ------- 39.3/48.9 MB 44.1 MB/s eta 0:00:01
   ---------------------------------------  48.8/48.9 MB 47.2 MB/s eta 0:00:01
   ---------------------------------------- 48.9/48.9 MB 44.2 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
# bow vs tfidf

# Import necessary libraries
import mlflow
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
import os

import dagshub

import dagshub
dagshub.init(repo_owner='abhityagi994-prog', repo_name='mlops-mini-project', mlflow=True)


# Load the data
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv').drop(columns=['tweet_id'])
df.head()

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(removing_numbers)
        df['content'] = df['content'].apply(removing_punctuations)
        df['content'] = df['content'].apply(removing_urls)
        df['content'] = df['content'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

# Normalize the text data
df = normalize_text(df)

x = df['sentiment'].isin(['happiness','sadness'])
df = df[x]

df['sentiment'] = df['sentiment'].replace({'sadness':0, 'happiness':1})

# Set the experiment name
mlflow.set_experiment("Bow vs TfIdf")

# Define feature extraction methods
vectorizers = {
    'BoW': CountVectorizer(),
    'TF-IDF': TfidfVectorizer()
}

# Define algorithms
algorithms = {
    'LogisticRegression': LogisticRegression(),
    'MultinomialNB': MultinomialNB(),
    'XGBoost': XGBClassifier(),
    'RandomForest': RandomForestClassifier(),
    'GradientBoosting': GradientBoostingClassifier()
}

# Start the parent run
with mlflow.start_run(run_name="All Experiments") as parent_run:
    # Loop through algorithms and feature extraction methods (Child Runs)
    for algo_name, algorithm in algorithms.items():
        for vec_name, vectorizer in vectorizers.items():
            with mlflow.start_run(run_name=f"{algo_name} with {vec_name}", nested=True) as child_run:
                X = vectorizer.fit_transform(df['content'])
                y = df['sentiment']
                X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
                

                # Log preprocessing parameters
                mlflow.log_param("vectorizer", vec_name)
                mlflow.log_param("algorithm", algo_name)
                mlflow.log_param("test_size", 0.2)
                
                # Model training
                model = algorithm
                model.fit(X_train, y_train)
                
                # Log model parameters
                if algo_name == 'LogisticRegression':
                    mlflow.log_param("C", model.C)
                elif algo_name == 'MultinomialNB':
                    mlflow.log_param("alpha", model.alpha)
                elif algo_name == 'XGBoost':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("learning_rate", model.learning_rate)
                elif algo_name == 'RandomForest':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("max_depth", model.max_depth)
                elif algo_name == 'GradientBoosting':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("learning_rate", model.learning_rate)
                    mlflow.log_param("max_depth", model.max_depth)
                
                # Model evaluation
                y_pred = model.predict(X_test)
                accuracy = accuracy_score(y_test, y_pred)
                precision = precision_score(y_test, y_pred)
                recall = recall_score(y_test, y_pred)
                f1 = f1_score(y_test, y_pred)
                
                # Log evaluation metrics
                mlflow.log_metric("accuracy", accuracy)
                mlflow.log_metric("precision", precision)
                mlflow.log_metric("recall", recall)
                mlflow.log_metric("f1_score", f1)
                
                # Log model
                import xgboost

                if isinstance(model, xgboost.XGBModel):
                    mlflow.xgboost.log_model(model, name="model")
                else:
                    mlflow.sklearn.log_model(
                        model,
                        name="model",
                        serialization_format="cloudpickle"
                    )
                
                # Save and log the notebook
                mlflow.log_artifact('exp1_bow_vs_tfidf.ipynb')
                
                # Print the results for verification
                print(f"Algorithm: {algo_name}, Feature Engineering: {vec_name}")
                print(f"Accuracy: {accuracy}")
                print(f"Precision: {precision}")
                print(f"Recall: {recall}")
                print(f"F1 Score: {f1}")

<>:60: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:60: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
C:\Users\Abhi\AppData\Local\Temp\ipykernel_20456\3701709914.py:60: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  text = re.sub('\s+', ' ', text).strip()


Initialized MLflow to track repo "abhityagi994-prog/mlops-mini-project"

Repository abhityagi994-prog/mlops-mini-project initialized!

C:\Users\Abhi\AppData\Local\Temp\ipykernel_20456\3701709914.py:88: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'] = df['sentiment'].replace({'sadness':0, 'happiness':1})
2026/08/29 00:53:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Algorithm: LogisticRegression, Feature Engineering: BoW
Accuracy: 0.7937349397590362
Precision: 0.7846750727449079
Recall: 0.7970443349753694
F1 Score: 0.7908113391984359
🏃 View run LogisticRegression with BoW at: https://dagshub.com/abhityagi994-prog/mlops-mini-project.mlflow/#/experiments/1/runs/a027904c3f674c46b454784b88420430
🧪 View experiment at: https://dagshub.com/abhityagi994-prog/mlops-mini-project.mlflow/#/experiments/1


...

Algorithm: RandomForest, Feature Engineering: TF-IDF
Accuracy: 0.771566265060241
Precision: 0.775178026449644
Recall: 0.7507389162561576
F1 Score: 0.7627627627627628
🏃 View run RandomForest with TF-IDF at: https://dagshub.com/abhityagi994-prog/mlops-mini-project.mlflow/#/experiments/1/runs/83cb8c9179d946cab730558f60fa274c
🧪 View experiment at: https://dagshub.com/abhityagi994-prog/mlops-mini-project.mlflow/#/experiments/1


2026/08/29 01:00:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Algorithm: GradientBoosting, Feature Engineering: BoW
Accuracy: 0.7272289156626506
Precision: 0.8013422818791947
Recall: 0.5881773399014778
F1 Score: 0.678409090909091
🏃 View run GradientBoosting with BoW at: https://dagshub.com/abhityagi994-prog/mlops-mini-project.mlflow/#/experiments/1/runs/84060aa0d30748fbb1bb2ef3ad002ec5
🧪 View experiment at: https://dagshub.com/abhityagi994-prog/mlops-mini-project.mlflow/#/experiments/1


2026/08/29 01:01:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Algorithm: GradientBoosting, Feature Engineering: TF-IDF
Accuracy: 0.723855421686747
Precision: 0.8044077134986226
Recall: 0.5753694581280788
F1 Score: 0.6708788052843193
🏃 View run GradientBoosting with TF-IDF at: https://dagshub.com/abhityagi994-prog/mlops-mini-project.mlflow/#/experiments/1/runs/54990f99a85d41fbb14e20ba8e2f0cb2
🧪 View experiment at: https://dagshub.com/abhityagi994-prog/mlops-mini-project.mlflow/#/experiments/1
🏃 View run All Experiments at: https://dagshub.com/abhityagi994-prog/mlops-mini-project.mlflow/#/experiments/1/runs/6b1bde3262ba4f809be2bb6c53f0002d
🧪 View experiment at: https://dagshub.com/abhityagi994-prog/mlops-mini-project.mlflow/#/experiments/1


In [6]:
import os

os.getcwd()

'c:\\Users\\Abhi\\Desktop\\mini-project\\mini-mlops-project\\notebooks'